<a href="https://colab.research.google.com/github/rachel-kim2255/Beer_Sentiment_Anlysis/blob/main/Kinton_Ramen_Data_Integration_%26_Cleaning_(Final).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Kinton Ramen Data Integration & Cleaning (Final)**

## 1. 데이터 통합 및 기본 전처리
여러 폴더에 흩어진 CSV 파일을 읽어와 매장명과 기간 정보를 추가합니다.

In [24]:
# ============================================================
# SECTION 1: DATA INTEGRATION
# ============================================================
# Load all store CSV files and merge into a single DataFrame.
# Added metadata columns: Store_Name, Data_Period, Year_Folder

import pandas as pd
import os
import glob
from google.colab import drive

drive.mount('/content/drive')

data_path = '/content/drive/MyDrive/KintonRamen/Kinton_raw_data/'
all_files = glob.glob(os.path.join(data_path, "**/*_transactions_*.csv"), recursive=True)

df_list = []

for filename in all_files:
    base_name = os.path.basename(filename)
    try:
        store_name = base_name.split('_')[1].replace('bc-f-', '')
        parts = base_name.split('_')
        period = f"{parts[-2]}_{parts[-1].replace('.csv', '')}"
    except IndexError:
        store_name, period = "Unknown", "Unknown"

    df = pd.read_csv(filename, encoding='utf-8-sig')
    df['Store_Name'] = store_name
    df['Data_Period'] = period
    df['Year_Folder'] = filename.split('/')[-2]
    df_list.append(df)

df_total = pd.concat(df_list, ignore_index=True)
print(f"Initial Integration Complete: {len(df_total):,} rows from {len(df_list)} files.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Initial Integration Complete: 2,702,816 rows from 21 files.


/tmp/ipykernel_6542/1056433184.py:34: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_total = pd.concat(df_list, ignore_index=True)


In [25]:
# ============================================================
# SECTION 2: DATA VERIFICATION
# ============================================================
# Verify the integrated dataset: row counts, date coverage,
# and distribution by store and year folder.

df_total['_date_temp'] = pd.to_datetime(
    df_total['Date'], format='%d/%m/%y %I:%M %p', errors='coerce'
)

# 2-1. Overall summary
print("=" * 60)
print(f"Total Rows: {len(df_total):,}")
print(f"Date Parse Failures: {df_total['_date_temp'].isna().sum():,}")

# 2-2. Coverage by store
print("\n" + "=" * 60)
print("Coverage by Store:")
coverage = df_total.groupby('Store_Name').agg(
    rows       = ('Store_Name', 'count'),
    start_date = ('_date_temp', 'min'),
    end_date   = ('_date_temp', 'max'),
).sort_values('start_date')
print(coverage.to_string())

# 2-3. Row count by store x year folder
print("\n" + "=" * 60)
print("Row Count by Store × Year Folder:")
print(df_total.groupby(['Store_Name', 'Year_Folder']).size().to_string())

# 2-4. Row count by store x data period
print("\n" + "=" * 60)
print("Row Count by Store × Data Period:")
print(df_total.groupby(['Store_Name', 'Data_Period']).size().to_string())

df_total = df_total.drop(columns=['_date_temp'])

Total Rows: 2,702,816
Date Parse Failures: 0

Coverage by Store:
                               rows          start_date            end_date
Store_Name                                                                 
vancouverubc                 469223 2025-06-10 09:35:00 2026-04-12 20:54:00
kelownadowntown              260806 2025-06-11 10:01:00 2026-04-12 22:46:00
vancouverrobson              584504 2025-06-11 10:07:00 2026-04-12 21:33:00
victoriadowntownwharfstreet   99975 2025-06-11 10:10:00 2026-04-12 20:53:00
vancouvermarinegateway       334926 2025-06-11 10:33:00 2026-04-12 22:04:00
surreykinggeorgehub          198718 2025-06-11 11:05:00 2026-04-12 22:43:00
vancouvernorthvancouver      438482 2025-06-11 11:12:00 2026-04-12 23:02:00
vancouverbroadway            159637 2025-12-11 16:29:00 2026-04-12 22:53:00
burnabygilmore                79333 2025-12-24 11:02:00 2026-04-12 21:53:00
coquitlamsunwoodsquare        50237 2026-02-06 12:03:00 2026-04-12 20:47:00
victoriauniversityheigh

In [26]:
# ============================================================
# SECTION 3: ANOMALY DETECTION
# ============================================================
# Two issues were identified during verification:
#
# [Issue 1] Date Boundary Overlap
#   Files named 20250610_20260101 and 20260101_20260413 both
#   include Jan 1, 2026 data → causes duplicate rows.
#
# [Issue 2] Shared Lightspeed Account (Identifier Collision)
#   Newly opened stores (Gilmore, Broadway) appear to have been
#   configured by copying an existing store's Lightspeed account.
#   This results in identical Identifiers appearing under
#   different Store_Names.
#   e.g. S1062188.1 appears in both 'kelownadowntown' AND
#        'burnabygilmore' with identical transaction content.

# --- Issue 1: Check date boundary overlap ---
print("=" * 60)
print("Issue 1: Checking for duplicate Identifiers")
dup_count = df_total.duplicated(subset=['Identifier']).sum()
print(f"Duplicate Identifiers (raw): {dup_count:,}")

# --- Issue 2: Cross-store Identifier collision evidence ---
print("\n" + "=" * 60)
print("Issue 2: Cross-store Identifier collision by store")
cross_dups = df_total[df_total.duplicated(subset=['Identifier'], keep=False)]
print(cross_dups.groupby('Store_Name').size().to_string())

# Evidence: same Identifier appearing in two different stores
print("\nEvidence — Identifier 'S1062188.1' in multiple stores:")
print(df_total[df_total['Identifier'] == 'S1062188.1'][
    ['Identifier', 'Date', 'Store_Name', 'Data_Period', 'Device_Name']
].to_string(index=False))

# Lightspeed account number extracted from Identifier
print("\n" + "=" * 60)
print("Lightspeed Account Number by Store:")
df_total['acct_num'] = df_total['Identifier'].str.extract(r'S(\d+)\.')
print(df_total.groupby('Store_Name')['acct_num'].unique().to_string())

Issue 1: Checking for duplicate Identifiers
Duplicate Identifiers (raw): 231,156

Issue 2: Cross-store Identifier collision by store
Store_Name
burnabygilmore                   6061
kelownadowntown                102597
surreykinggeorgehub            111172
vancouverbroadway               14636
vancouvermarinegateway          57960
vancouvernorthvancouver         60048
vancouverrobson                  2088
vancouverubc                    26900
victoriadowntownwharfstreet     53875
victoriauniversityheights       26975

Evidence — Identifier 'S1062188.1' in multiple stores:
Identifier              Date      Store_Name       Data_Period      Device_Name
S1062188.1 24/12/25 11:02 AM kelownadowntown 20250610_20260101 iPad222(1062188)
S1062188.1 24/12/25 11:02 AM  burnabygilmore 20251201_20260101 iPad222(1062188)

Lightspeed Account Number by Store:
Store_Name
burnabygilmore                                        [1062188, 1094487]
coquitlamsunwoodsquare                                     

In [27]:
# 날짜 파싱 후 재확인
df_total['_date_temp'] = pd.to_datetime(
    df_total['Date'], format='%d/%m/%y %I:%M %p', errors='coerce'
)

shared_accts = ['992100', '992127', '992113', '992096', '991561', '992112']

for acct in shared_accts:
    print(f"\n=== Account {acct} ===")
    result = df_total[df_total['acct_num'] == acct].groupby('Store_Name').agg(
        rows  = ('Identifier', 'count'),
        start = ('_date_temp', 'min'),
        end   = ('_date_temp', 'max')
    )
    print(result.to_string())

df_total = df_total.drop(columns=['_date_temp'])


=== Account 992100 ===
                       rows               start                 end
Store_Name                                                         
kelownadowntown      113744 2026-01-30 11:55:00 2026-04-12 22:46:00
surreykinggeorgehub  175999 2025-06-11 11:05:00 2026-04-01 20:52:00

=== Account 992127 ===
                          rows               start                 end
Store_Name                                                            
vancouverbroadway       123000 2026-01-22 17:48:00 2026-04-12 22:53:00
vancouvermarinegateway   48000 2025-06-11 10:33:00 2026-01-13 11:38:00

=== Account 992113 ===
                           rows               start                 end
Store_Name                                                             
vancouvermarinegateway   286926 2025-06-30 12:16:00 2026-04-12 22:04:00
vancouvernorthvancouver   96000 2025-06-11 11:12:00 2026-01-19 17:00:00

=== Account 992096 ===
                           rows               start         

In [34]:
# 전 매장 월별 데이터 존재 여부 한번에 확인
df_total['_date_temp'] = pd.to_datetime(
    df_total['Date'], format='%d/%m/%y %I:%M %p', errors='coerce'
)

monthly_check = df_total.groupby(
    ['Store_Name', df_total['_date_temp'].dt.to_period('M')]
).size().unstack(fill_value=0)

print(monthly_check.to_string())

df_total = df_total.drop(columns=['_date_temp'])

_date_temp                   2025-06  2025-07  2025-08  2025-09  2025-10  2025-11  2025-12  2026-01  2026-02  2026-03  2026-04
Store_Name                                                                                                                    
burnabygilmore                     0        0        0        0        0        0     6061    22265    23192    19799     8016
coquitlamsunwoodsquare             0        0        0        0        0        0        0        0    11639    27374    11224
kelownadowntown                18771    29354    32178    26697        0        0     6061    38164    46223    44897    18461
surreykinggeorgehub            25758     4242        0        0        0        0    14636    53628    46223    44897     9334
vancouverbroadway                  0        0        0        0        0        0    14636    36991    43686    45320    19004
vancouvermarinegateway         30749    23920    28256    32138    31434    33736    37323    43448    30190   

In [28]:
# ============================================================
# SECTION 4: DEDUPLICATION
# ============================================================
# Resolution strategy:
#
# Use (Store_Name + Identifier) as the composite unique key.
# - Fixes Issue 1: same Identifier in same store from
#   overlapping date-range files → keep first occurrence.
# - Fixes Issue 2: same Identifier across different stores
#   → treated as distinct records (different Store_Name).

before = len(df_total)
df_total = df_total.drop_duplicates(
    subset=['Store_Name', 'Identifier'], keep='first'
)
after = len(df_total)

print(f"Before deduplication: {before:,} rows")
print(f"After deduplication:  {after:,} rows")
print(f"Rows removed:         {before - after:,} rows")

Before deduplication: 2,702,816 rows
After deduplication:  2,702,816 rows
Rows removed:         0 rows


In [29]:
# ============================================================
# SECTION 5: POST-DEDUPLICATION VERIFICATION
# ============================================================
# Confirm that all stores are present and no duplicates remain.

# 5-1. Remaining duplicates
dup_remaining = df_total.duplicated(subset=['Store_Name', 'Identifier']).sum()
print(f"Remaining duplicates: {dup_remaining:,}")

# 5-2. Store list and row counts
print("\nFinal row count by store:")
print(df_total.groupby('Store_Name')['Identifier'].count()
      .sort_values(ascending=False).to_string())

# 5-3. Date coverage confirmation
df_total['_date_temp'] = pd.to_datetime(
    df_total['Date'], format='%d/%m/%y %I:%M %p', errors='coerce'
)
print("\nFinal date coverage by store:")
print(df_total.groupby('Store_Name').agg(
    rows       = ('Store_Name', 'count'),
    start_date = ('_date_temp', 'min'),
    end_date   = ('_date_temp', 'max'),
).sort_values('start_date').to_string())

df_total = df_total.drop(columns=['_date_temp', 'acct_num'])
print("\nIntegration complete. df_total is ready for cleaning.")

Remaining duplicates: 0

Final row count by store:
Store_Name
vancouverrobson                584504
vancouverubc                   469223
vancouvernorthvancouver        438482
vancouvermarinegateway         334926
kelownadowntown                260806
surreykinggeorgehub            198718
vancouverbroadway              159637
victoriadowntownwharfstreet     99975
burnabygilmore                  79333
coquitlamsunwoodsquare          50237
victoriauniversityheights       26975

Final date coverage by store:
                               rows          start_date            end_date
Store_Name                                                                 
vancouverubc                 469223 2025-06-10 09:35:00 2026-04-12 20:54:00
kelownadowntown              260806 2025-06-11 10:01:00 2026-04-12 22:46:00
vancouverrobson              584504 2025-06-11 10:07:00 2026-04-12 21:33:00
victoriadowntownwharfstreet   99975 2025-06-11 10:10:00 2026-04-12 20:53:00
vancouvermarinegateway       3349

## 2. Troubleshooting & Discovery Log

In [30]:
# """
# [TROUBLESHOOTING LOG]
# 1. Discovery of Overlaps:
#    - Noticed Jan 1st data was present in both '20251201_20260101' and '20260101_20260413' files.

# 2. Discovery of Cross-Store ID Conflict:
#    - Initial deduplication by 'Identifier' only resulted in 10 stores (Victoria University Heights disappeared).
#    - Investigation revealed that iPad devices (e.g., iPad222) were moved between stores without resetting
#      internal IDs, causing identical Identifiers across different locations (e.g., Burnaby vs Kelowna).

# 3. Solution:
#    - Deduplication must be performed using a composite key: ['Store_Name', 'Identifier'].
# """

# print("\n" + "=" * 60)
# print("Handling overlaps and cross-store ID conflicts...")

# before_len = len(df_total)

# # ValueError 방지를 위해 계산 후 f-string 적용
# df_total = df_total.drop_duplicates(subset=['Store_Name', 'Identifier'], keep='first')

# after_len = len(df_total)
# removed_len = before_len - after_len

# # f-string formatting 수정 (ValueError: Cannot specify ',' with 's' 해결)
# print(f"Total rows before: {before_len:,}")
# print(f"Total rows after:  {after_len:,}")
# print(f"Rows removed:      {removed_len:,}") # 숫자로 확실히 계산 후 출력

## 2. 중복 제거
여기서 발견한 **'매장 간 Identifier 중복 문제'**를 해결하는 것이 가장 중요합니다.

Identifier는 매장을 구별하는 식별자로 설정됩니다. 그러나 새로운 매장이 오픈하면서 타 매장과 중복되는 Identifier를 사용하는 것이 발견되었습니다. 또한 1월 1일자 데이터가 중복으로 발견되었습니다.

In [31]:
# # Date parsing for coverage analysis
# df_total['Date_Parsed'] = pd.to_datetime(df_total['Date'], format='%d/%m/%y %I:%M %p', errors='coerce')

# print("\n" + "=" * 60)
# print(f"Final Unique Store Count: {df_total['Store_Name'].nunique()}")
# print("=" * 60)

# # Verify all 11 stores are present
# store_summary = df_total.groupby('Store_Name').agg(
#     Rows = ('Identifier', 'count'),
#     Min_Date = ('Date_Parsed', 'min'),
#     Max_Date = ('Date_Parsed', 'max')
# ).sort_values('Min_Date')

# display(store_summary)

In [32]:
# save_path = '/content/drive/MyDrive/KintonRamen/Kinton_raw_data/Kinton_Refined_Data/'
# if not os.path.exists(save_path): os.makedirs(save_path)

# # 용량이 크므로 Parquet 포맷 권장 (CSV보다 빠르고 가벼움)
# df_total.to_parquet(os.path.join(save_path, 'kinton_bc_11_stores_cleaned.parquet'), index=False)
# print(f"Success! Cleaned data saved to {save_path}")